This is a vectorized version of the environment. How does it work? Simply put, instead of running a single environment, we're running them into batches (vectorization). At each time step, instead of formulating a single action $a$, we'll define it as a vector $a=[0, .., n_\text{envs}]$ where each entry corresponds to the action to be performed for an environment.

Thus, if you run 4 simulatenous environments, your observations space becomes (4, num_observations). Because we use a step size of 10 and have a total of 9 snesors, each observation will result into a space of (40,9) elements. The core idea is to speed up the inference and training time of the model instead of querying a single environment.

In our implementation, based on what model you decide, you should take as inputs the observations reshaped to your liking, and predict the actions $a$ from your policy.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
from student_client.student_gym_env_vectorized import create_student_gym_env_vectorized

env = create_student_gym_env_vectorized(
            num_envs=4,
            user_token='SERgio26735540'
        )

2026-03-02 22:53:28,600 - student_client.student_gym_env_vectorized - WARNING - No .env file found and no explicit parameters provided. Using default values. For better setup, create a .env file with:
SERVER_URL=http://rlchallenge.orailix.com
USER_TOKEN=student_user
ENV_TYPE=DegradationEnv
MAX_STEPS_PER_EPISODE=1000
AUTO_RESET=True
TIMEOUT=30.0


2026-03-02 22:53:28,670 - httpx - INFO - HTTP Request: GET http://rlchallenge.orailix.com/api/v1/version "HTTP/1.1 200 OK"
2026-03-02 22:53:28,671 - student_client.student_gym_env_vectorized - INFO - Client is up to date (version 0.4)
2026-03-02 22:53:28,701 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/session/create "HTTP/1.1 200 OK"
2026-03-02 22:53:28,702 - student_client.student_gym_env_vectorized - INFO - Created new session: c8e8ebcc-14d9-4f1f-84f1-7ae50348fbc6
2026-03-02 22:53:29,560 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/vectorized/episodes/create "HTTP/1.1 200 OK"
2026-03-02 22:53:29,561 - student_client.student_gym_env_vectorized - INFO - Created vectorized group with 4 episodes
2026-03-02 22:53:29,562 - student_client.student_gym_env_vectorized - INFO - Vectorized group ID: 749950c1-7e2a-447a-a3d2-3c398a189d38
2026-03-02 22:53:29,563 - student_client.student_gym_env_vectorized - INFO - StudentGymEnvVectorized in

In [4]:
print(f"Environment created with {env.num_envs} parallel environments")
print(f"   Episode IDs: {env.episode_ids}")

# Reset all environments
print(f"\n🔄 Resetting all environments...")
observations, infos = env.reset()

print(f"   Observations shape: {observations.shape}")
print(f"   First observation: {observations[0]}")

Environment created with 4 parallel environments
   Episode IDs: ['cad3b288-59c7-461d-aef7-d802848f4e5a', '6447b0aa-3171-4046-8864-586c9626e029', 'e9c3a991-5de9-45ea-866f-f44ce9243c87', 'ad3356be-7ee4-4a0a-a672-bacef3926606']

🔄 Resetting all environments...


2026-03-02 22:53:36,024 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_reset "HTTP/1.1 200 OK"
2026-03-02 22:53:36,025 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 0 episode ID changed from cad3b288-59c7-461d-aef7-d802848f4e5a to 4afd7d1a-871b-4f55-8de7-34f73c2520d5
2026-03-02 22:53:36,026 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 1 episode ID changed from 6447b0aa-3171-4046-8864-586c9626e029 to 6796c1f4-0f0b-4129-a029-cf4a4d565f2d
2026-03-02 22:53:36,026 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 2 episode ID changed from e9c3a991-5de9-45ea-866f-f44ce9243c87 to f85b8e47-c8cf-43e5-a6c1-b12c4ea2fd93
2026-03-02 22:53:36,027 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 3 episode ID changed from ad3356be-7ee4-4a0a-a672-bacef3926606 to 14e61b58-37c7-49b2-8d73-9123fa2fc89a
2026-03-02 22:53:36,027 - student_client.student_gym_env_vectorized - INFO - All 4 

   Observations shape: (4, 9)
   First observation: [7.9778314e+02 1.9436205e+04 3.3608740e+02 1.1219043e+03 3.7243864e-01
 1.3664049e+06 3.9592202e+03 0.0000000e+00 1.0068630e+01]


## Training / Iterations

Here you can iterate through the vectorized environments. You'll notice that actions are a vector where each entry corresponds to the associated environment in the vector.

In A), we automatically reset the envs that have terminated so you can continue for an indefinite amount of steps. As environments don't have the same length, they stop at different times, this helps you reset terminated episodes on the fly.

Tips:
- The step_size return many observations, should you feed each one-by-one in your model, or the full step_size=10 one? The choice is yours!
- There exists multiple ways of exploring the dataset

In [5]:
for step in range(40):

    # A) Check if any environments terminated
    terminated_envs = env.get_terminated_env_indices()
    if terminated_envs:

        print(f"   ⚠️  Environments {terminated_envs} terminated")
        reset_obs, reset_infos = env.reset_specific_envs(terminated_envs)
        for i, env_id in enumerate(terminated_envs):
            infos[env_id] = reset_infos[i] # reset previous info dict

        #break

    # Generate random actions for all environments
    actions = np.random.randint(1, 3, size=env.num_envs)
    #actions = np.asarray([0,0,0,0])

    print(f"\n   Step {step + 1}:")
    print(f"      Actions: {actions}")

    # Take step
    observations, rewards, terminateds, truncateds, infos = env.step(actions)

    print(f"      Rewards: {rewards}")
    print(f"      Terminated: {terminateds}")
    for i, obs in enumerate(observations):
        print(f"      Obs {i}: {obs.shape}")
    print(f"      Active environments: {env.get_active_count()}/{env.num_envs}")

    # Show filtered info (production mode)
    for i, info in enumerate(infos):
        print(f"      Env {i} info: {info}")

    #break

#env.close()


   Step 1:
      Actions: [1 2 1 2]


2026-03-02 22:53:43,794 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_step "HTTP/1.1 200 OK"


      Rewards: [-531.8017   150.      -491.99948  150.     ]
      Terminated: [False  True False  True]
      Obs 0: (10, 9)
      Obs 1: (1, 9)
      Obs 2: (10, 9)
      Obs 3: (1, 9)
      Active environments: 2/4
      Env 0 info: {'step': 10, 'terminated': False, 'truncated': False}
      Env 1 info: {'step': 1, 'terminated': True, 'truncated': False}
      Env 2 info: {'step': 10, 'terminated': False, 'truncated': False}
      Env 3 info: {'step': 1, 'terminated': True, 'truncated': False}
   ⚠️  Environments [1, 3] terminated


2026-03-02 22:53:45,910 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_reset "HTTP/1.1 200 OK"
2026-03-02 22:53:45,911 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 1 episode ID changed from 6796c1f4-0f0b-4129-a029-cf4a4d565f2d to c5137273-85d5-4879-ba48-a844b7ed176a
2026-03-02 22:53:45,912 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 3 episode ID changed from 14e61b58-37c7-49b2-8d73-9123fa2fc89a to 185b68f9-5bba-4378-b150-af40acfe6942



   Step 2:
      Actions: [2 2 2 2]


...

      Rewards: [ 150.       150.       150.      -530.71625]
      Terminated: [ True  True  True False]
      Obs 0: (1, 9)
      Obs 1: (1, 9)
      Obs 2: (1, 9)
      Obs 3: (10, 9)
      Active environments: 1/4
      Env 0 info: {'step': 11, 'terminated': True, 'truncated': False}
      Env 1 info: {'step': 21, 'terminated': True, 'truncated': False}
      Env 2 info: {'step': 11, 'terminated': True, 'truncated': False}
      Env 3 info: {'step': 30, 'terminated': False, 'truncated': False}
   ⚠️  Environments [0, 1, 2] terminated


2026-03-02 22:57:59,763 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_reset "HTTP/1.1 200 OK"
2026-03-02 22:57:59,764 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 0 episode ID changed from 3a3bb6b6-deb6-40fa-a3ec-c2296f196276 to 9092becc-a7db-4eb7-868b-dd1e1b8e5738
2026-03-02 22:57:59,765 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 1 episode ID changed from 42c0fe5e-da9c-4e5a-95f1-01b29d930f72 to 5f9ef40e-5d68-4425-9a0c-e9fcc11f917a
2026-03-02 22:57:59,766 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 2 episode ID changed from 220ac9cf-5c21-48ad-817c-f0e940dcebb1 to d2df1133-73e2-4c48-9db6-75543b9a1dc9



   Step 40:
      Actions: [2 1 2 1]


2026-03-02 22:58:04,558 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_step "HTTP/1.1 200 OK"


      Rewards: [ 150.      -526.05396  150.      -498.39032]
      Terminated: [ True False  True False]
      Obs 0: (1, 9)
      Obs 1: (10, 9)
      Obs 2: (1, 9)
      Obs 3: (10, 9)
      Active environments: 2/4
      Env 0 info: {'step': 1, 'terminated': True, 'truncated': False}
      Env 1 info: {'step': 10, 'terminated': False, 'truncated': False}
      Env 2 info: {'step': 1, 'terminated': True, 'truncated': False}
      Env 3 info: {'step': 40, 'terminated': False, 'truncated': False}


In [46]:
for obs_env in observations:
    print(obs_env.shape)

(1, 9)
(1, 9)
(1, 9)
(1, 9)


In [51]:
observations

[array([[7.9119489e+02, 1.9313773e+04, 3.3493365e+02, 1.1183016e+03,
         3.7162682e-01, 1.3504325e+06, 3.9524648e+03, 0.0000000e+00,
         9.1000004e+00]], dtype=float32),
 array([[7.9269122e+02, 1.9191648e+04, 3.3450665e+02, 1.1146975e+03,
         3.7097099e-01, 1.3704700e+06, 3.9508916e+03, 0.0000000e+00,
         9.2736034e+00]], dtype=float32),
 array([[7.9539990e+02, 1.9358600e+04, 3.3558124e+02, 1.1196388e+03,
         3.7197024e-01, 1.3640279e+06, 3.9565410e+03, 0.0000000e+00,
         9.4699183e+00]], dtype=float32),
 array([[7.9565344e+02, 1.9362465e+04, 3.3514899e+02, 1.1180753e+03,
         3.7165123e-01, 1.3695479e+06, 3.9540964e+03, 0.0000000e+00,
         9.7564869e+00]], dtype=float32)]